# GraphSentry — Complete Experiment Pipeline

**A Unified GNN Framework for Blockchain Illicit Activity Detection**

This notebook contains the entire experimental pipeline:
1. **Data Pipeline** — Elliptic2 dataset loading, feature extraction, stratified split
2. **Model Training** — GraphSAINT grid search (GCN/GAT × node/edge/rw)
3. **Baselines** — GCN/GAT/SAGE with standard DataLoader
4. **Ablations** — Feature, pooling, depth, loss, and residual ablations
5. **Statistical Significance** — 5-seed paired t-tests
6. **Full-Scale Training** — 121K CCs from raw Elliptic2 CSVs

**Architecture:** 2-layer GNN with residual connections, Global Max Pool, focal loss,
43-dim anonymous financial features (no engineered degree feature).

**Environment:** Google Colab (T4 GPU, 16GB VRAM, ~12GB system RAM)

## 0. Configuration

In [57]:
SUBSET_MODE = True
SUBSET_N_ILLICIT = 2763
SUBSET_N_LICIT = 25000
MIN_CC_SIZE = 1

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

CHUNK_SIZE = 500_000
N_ANONYMOUS_FEATURES = 43

MAX_EPOCHS = 60
PATIENCE = 10
BATCH_SIZE = 64
LR = 0.005
LR_FACTOR = 0.5
LR_PATIENCE = 5
HIDDEN_DIM = 128
DROPOUT = 0.5
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.75

SAINT_BUDGET = 500
SAINT_SAMPLES_PER_EPOCH = 20
STRATEGIES = ['node', 'edge', 'rw']
BACKBONES = ['GCN', 'GAT']
SEEDS = [42, 123, 456, 789, 1024]
SEED = 42

BASE_PATH = '/content/drive/MyDrive/GraphSentry'
RAW_PATH = f'{BASE_PATH}/data/raw'
PROCESSED_PATH = f'{BASE_PATH}/data/processed'

FEATURE_FILE = f'{RAW_PATH}/background_nodes.csv'
LABELS_FILE = f'{RAW_PATH}/connected_components.csv'
NODES_FILE = f'{RAW_PATH}/nodes.csv'
EDGES_FILE = f'{RAW_PATH}/edges.csv'

## 0.1 Environment setup

In [58]:
!pip install -q uv
!uv pip install --system torch torch-geometric numpy pandas networkx tqdm scikit-learn scipy psutil
!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html

from google.colab import drive
import os, json, pickle, time, random, copy, warnings

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import networkx as nx
import psutil
from torch_geometric.data import Data, Batch
from torch_geometric.nn import (
    GCNConv, GATConv, SAGEConv,
    global_mean_pool, global_max_pool, global_add_pool,
)
from torch_geometric.utils import degree
from torch_geometric.loader import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    confusion_matrix, average_precision_score,
)
from scipy import stats as scipy_stats
from tqdm import tqdm

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

os.makedirs(PROCESSED_PATH, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f"Mode: {'SUBSET' if SUBSET_MODE else 'FULL SCALE'}")

Using Python 3.12.13 environment at: /usr
Checked 9 packages in 87ms
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Mode: SUBSET


# Part 1: Data Pipeline

## 1.1 Load raw data

In [59]:
t0 = time.time()

df_labels = pd.read_csv(LABELS_FILE)
df_labels['label'] = (df_labels['ccLabel'] != 'licit').astype(int)

total_illicit = df_labels['label'].sum()
total_licit = len(df_labels) - total_illicit
print(f'Elliptic2 CCs: {len(df_labels):,} total ({total_illicit:,} illicit, {total_licit:,} licit)')

df_nodes_raw = pd.read_csv(NODES_FILE)
print(f"Elliptic2 nodes: {df_nodes_raw['clId'].nunique():,}")

df_edges_raw = pd.read_csv(EDGES_FILE)
print(f'Elliptic2 edges: {len(df_edges_raw):,}')

print(f'\nCSVs loaded in {time.time() - t0:.1f}s')

Elliptic2 CCs: 121,810 total (2,763 illicit, 119,047 licit)
Elliptic2 nodes: 444,521
Elliptic2 edges: 367,137

CSVs loaded in 0.3s


## 1.2 CC selection

In [60]:
cc_sizes_df = df_nodes_raw.groupby('ccId').size().reset_index(name='size')
df_labels_with_size = df_labels.merge(cc_sizes_df, on='ccId', how='left')
df_labels_with_size = df_labels_with_size[df_labels_with_size['size'] >= MIN_CC_SIZE]

if SUBSET_MODE:
    avail_illicit = df_labels_with_size[df_labels_with_size['label'] == 1]
    avail_licit = df_labels_with_size[df_labels_with_size['label'] == 0]

    illicit_ccs = avail_illicit.sample(
        n=min(SUBSET_N_ILLICIT, len(avail_illicit)), random_state=SEED
    )
    licit_ccs = avail_licit.sample(
        n=min(SUBSET_N_LICIT, len(avail_licit)), random_state=SEED
    )
    selected_ccs = pd.concat([illicit_ccs, licit_ccs]).sample(frac=1, random_state=SEED)
else:
    selected_ccs = df_labels_with_size[df_labels_with_size['label'].notna()].copy()

target_cc_ids = set(selected_ccs['ccId'].values)
cc_label_map = dict(zip(selected_ccs['ccId'], selected_ccs['label']))

n_illicit = int(selected_ccs['label'].sum())
n_licit = len(selected_ccs) - n_illicit
print(f'Available CCs (size >= {MIN_CC_SIZE}): {len(df_labels_with_size):,} '
      f'({int(df_labels_with_size["label"].sum()):,} illicit, '
      f'{len(df_labels_with_size) - int(df_labels_with_size["label"].sum()):,} licit)')
print(f'Selected CCs: {len(selected_ccs):,} ({n_illicit:,} illicit, {n_licit:,} licit)')
print(f'Class ratio: 1:{n_licit / max(n_illicit, 1):.1f} (illicit:licit)')

Available CCs (size >= 1): 121,810 (2,763 illicit, 119,047 licit)
Selected CCs: 27,763 (2,763 illicit, 25,000 licit)
Class ratio: 1:9.0 (illicit:licit)


## 1.3 Build background graph

In [61]:
t0 = time.time()

df_nodes = df_nodes_raw[df_nodes_raw['ccId'].isin(target_cc_ids)].copy()
df_nodes = df_nodes.merge(selected_ccs[['ccId', 'label']], on='ccId')

unique_nodes = np.sort(df_nodes['clId'].unique())
node_to_idx = {int(nid): i for i, nid in enumerate(unique_nodes)}
N = len(unique_nodes)

node_to_cc = {}
cc_to_nodes = {}

for cc_id, group in df_nodes.groupby('ccId'):
    node_indices = [node_to_idx[nid] for nid in group['clId'].values]
    cc_to_nodes[cc_id] = sorted(node_indices)
    for idx in node_indices:
        node_to_cc[idx] = cc_id

node_id_set = set(unique_nodes)
edge_mask = df_edges_raw['clId1'].isin(node_id_set) & df_edges_raw['clId2'].isin(node_id_set)
df_edges = df_edges_raw[edge_mask]

src = np.array([node_to_idx[n] for n in df_edges['clId1'].values])
dst = np.array([node_to_idx[n] for n in df_edges['clId2'].values])
bg_edge_index = torch.tensor(np.stack([src, dst]), dtype=torch.long)

adj_list = [[] for _ in range(N)]
for s, d in zip(src, dst):
    adj_list[s].append(d)
    adj_list[d].append(s)

M = bg_edge_index.size(1)
cc_sizes = [len(nodes) for nodes in cc_to_nodes.values()]

print(f'Background graph constructed in {time.time() - t0:.1f}s')
print(f'  Nodes: {N:,}')
print(f'  Edges: {M:,}')
print(f'  CCs:   {len(cc_to_nodes):,}')
print(f'  CC sizes: min={min(cc_sizes)}, max={max(cc_sizes)}, '
      f'mean={np.mean(cc_sizes):.1f}, median={np.median(cc_sizes):.0f}')

Background graph constructed in 1.3s
  Nodes: 101,955
  Edges: 83,858
  CCs:   27,763
  CC sizes: min=2, max=296, mean=3.7, median=3


## 1.4 Feature extraction (43 dims)

In [62]:
t0 = time.time()
features_anonymous = torch.zeros((N, N_ANONYMOUS_FEATURES), dtype=torch.float)
nodes_found = 0

chunk_iter = pd.read_csv(FEATURE_FILE, chunksize=CHUNK_SIZE)
for chunk in tqdm(chunk_iter, desc='Loading features'):
    mask = chunk['clId'].isin(node_id_set)
    matched = chunk[mask]
    if len(matched) == 0:
        continue

    feat_cols = [c for c in matched.columns if c.startswith('feat')]
    for _, row in matched.iterrows():
        idx = node_to_idx.get(row['clId'])
        if idx is not None:
            features_anonymous[idx] = torch.tensor(
                row[feat_cols].values.astype(np.float32)
            )
            nodes_found += 1

coverage = nodes_found / N * 100
print(f'Features loaded in {time.time() - t0:.1f}s')
print(f'  Shape: {features_anonymous.shape}')
print(f'  Coverage: {nodes_found:,} / {N:,} nodes ({coverage:.1f}%)')
print(f'  Non-zero rows: {(features_anonymous != 0).any(dim=1).sum().item():,}')

Loading features: 99it [02:53,  1.75s/it]

Features loaded in 173.9s
  Shape: torch.Size([101955, 43])
  Coverage: 101,955 / 101,955 nodes (100.0%)
  Non-zero rows: 101,955


## 1.5 Train / validation / test split

In [63]:
all_cc_ids = np.array(sorted(cc_to_nodes.keys()))
all_labels = np.array([cc_label_map[cc] for cc in all_cc_ids])

train_ids, temp_ids, train_labels, temp_labels = train_test_split(
    all_cc_ids, all_labels,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=all_labels,
    random_state=SEED,
)

relative_test = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
val_ids, test_ids, val_labels, test_labels = train_test_split(
    temp_ids, temp_labels,
    test_size=relative_test,
    stratify=temp_labels,
    random_state=SEED,
)

split_map = {}
for cc_id in train_ids:
    split_map[cc_id] = 'train'
for cc_id in val_ids:
    split_map[cc_id] = 'val'
for cc_id in test_ids:
    split_map[cc_id] = 'test'

def _split_summary(ids, labels, name):
    n_ill = labels.sum()
    return f'  {name:6s}: {len(ids):>6,} CCs ({n_ill:,} illicit, {len(ids) - n_ill:,} licit)'

print('Split summary:')
print(_split_summary(train_ids, train_labels, 'Train'))
print(_split_summary(val_ids, val_labels, 'Val'))
print(_split_summary(test_ids, test_labels, 'Test'))

train_node_indices = []
for cc_id in train_ids:
    train_node_indices.extend(cc_to_nodes[cc_id])
train_node_indices = np.array(train_node_indices)

anon_train = features_anonymous[train_node_indices].numpy()
valid_mask = (anon_train != 0).any(axis=1)
scaler_anonymous = StandardScaler()
scaler_anonymous.fit(anon_train[valid_mask])

features_anonymous_norm = torch.tensor(
    scaler_anonymous.transform(features_anonymous.numpy()), dtype=torch.float
)

print(f'\nScaler fitted on {len(train_node_indices):,} training nodes ({valid_mask.sum():,} valid rows)')

Split summary:
  Train : 19,434 CCs (1,934 illicit, 17,500 licit)
  Val   :  4,164 CCs (414 illicit, 3,750 licit)
  Test  :  4,165 CCs (415 illicit, 3,750 licit)

Scaler fitted on 71,700 training nodes (71,700 valid rows)


## 1.6 Save artefacts

In [64]:
t0 = time.time()

cc_ids_ordered = sorted(cc_to_nodes.keys())

torch.save({
    'edge_index': bg_edge_index,
    'adj_list': adj_list,
    'node_to_idx': node_to_idx,
    'node_to_cc': node_to_cc,
    'cc_to_nodes': cc_to_nodes,
    'num_nodes': N,
    'num_edges': M,
}, os.path.join(PROCESSED_PATH, 'background_graph.pt'))

torch.save({
    'cc_label_map': cc_label_map,
    'split_map': split_map,
    'train_ids': train_ids.tolist(),
    'val_ids': val_ids.tolist(),
    'test_ids': test_ids.tolist(),
    'cc_ids_ordered': [int(x) for x in cc_ids_ordered],
}, os.path.join(PROCESSED_PATH, 'cc_metadata.pt'))

torch.save(features_anonymous_norm, os.path.join(PROCESSED_PATH, 'features_anonymous.pt'))

with open(os.path.join(PROCESSED_PATH, 'scaler_anonymous.pkl'), 'wb') as f:
    pickle.dump(scaler_anonymous, f)

stats = {
    'subset_mode': SUBSET_MODE,
    'seed': SEED,
    'min_cc_size': MIN_CC_SIZE,
    'total_ccs': len(cc_to_nodes),
    'total_nodes': N,
    'total_edges': M,
    'n_illicit': int(n_illicit),
    'n_licit': int(n_licit),
    'class_ratio': round(n_licit / max(n_illicit, 1), 1),
    'split': {
        'train': len(train_ids),
        'val': len(val_ids),
        'test': len(test_ids),
    },
    'split_illicit': {
        'train': int(train_labels.sum()),
        'val': int(val_labels.sum()),
        'test': int(test_labels.sum()),
    },
    'cc_sizes': {
        'min': int(min(cc_sizes)),
        'max': int(max(cc_sizes)),
        'mean': round(float(np.mean(cc_sizes)), 1),
        'median': int(np.median(cc_sizes)),
    },
    'feature_dims': {
        'anonymous': N_ANONYMOUS_FEATURES,
    },
}

with open(os.path.join(PROCESSED_PATH, 'dataset_stats.json'), 'w') as f:
    json.dump(stats, f, indent=2)

print(f'Artefacts saved to {PROCESSED_PATH} in {time.time() - t0:.1f}s\n')
for fname in sorted(os.listdir(PROCESSED_PATH)):
    fpath = os.path.join(PROCESSED_PATH, fname)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f'  {fname:<30s} {size_mb:>8.2f} MB')

Artefacts saved to /content/drive/MyDrive/GraphSentry/data/processed in 1.6s

  background_graph.pt               12.36 MB
  cc_metadata.pt                     1.79 MB
  dataset_stats.json                 0.00 MB
  features_anonymous.pt             16.73 MB
  scaler_anonymous.pkl               0.00 MB


## 1.7 Validation

In [65]:
print('=' * 60)
print('VALIDATION: Reloading artefacts from disk')
print('=' * 60)

bg = torch.load(os.path.join(PROCESSED_PATH, 'background_graph.pt'), weights_only=False)
meta = torch.load(os.path.join(PROCESSED_PATH, 'cc_metadata.pt'), weights_only=False)
feat_anon = torch.load(os.path.join(PROCESSED_PATH, 'features_anonymous.pt'), weights_only=False)

with open(os.path.join(PROCESSED_PATH, 'scaler_anonymous.pkl'), 'rb') as f:
    sc_anon = pickle.load(f)
with open(os.path.join(PROCESSED_PATH, 'dataset_stats.json'), 'r') as f:
    loaded_stats = json.load(f)

assert bg['edge_index'].shape[0] == 2
assert feat_anon.shape == (bg['num_nodes'], N_ANONYMOUS_FEATURES)
assert len(bg['adj_list']) == bg['num_nodes']

all_split_ids = set(meta['train_ids'] + meta['val_ids'] + meta['test_ids'])
assert len(all_split_ids) == len(bg['cc_to_nodes'])

train_set = set(meta['train_ids'])
val_set = set(meta['val_ids'])
test_set = set(meta['test_ids'])
assert len(train_set & val_set) == 0
assert len(train_set & test_set) == 0
assert len(val_set & test_set) == 0

assert sc_anon.n_features_in_ == N_ANONYMOUS_FEATURES

print('\nAll checks passed.\n')
print(json.dumps(loaded_stats, indent=2))

VALIDATION: Reloading artefacts from disk

All checks passed.

{
  "subset_mode": true,
  "seed": 42,
  "min_cc_size": 1,
  "total_ccs": 27763,
  "total_nodes": 101955,
  "total_edges": 83858,
  "n_illicit": 2763,
  "n_licit": 25000,
  "class_ratio": 9.0,
  "split": {
    "train": 19434,
    "val": 4164,
    "test": 4165
  },
  "split_illicit": {
    "train": 1934,
    "val": 414,
    "test": 415
  },
  "cc_sizes": {
    "min": 2,
    "max": 296,
    "mean": 3.7,
    "median": 3
  },
  "feature_dims": {
    "anonymous": 43
  }
}


# Part 2: Shared Infrastructure

All model architectures, the GraphSAINT sampler, loss functions, and
evaluation utilities are defined here once and reused by all subsequent parts.

## 2.1 Graph construction

In [66]:
def build_pyg_graph(cc_id):
    cc_nodes = sorted(bg['cc_to_nodes'][cc_id])
    local_map = {g: l for l, g in enumerate(cc_nodes)}
    adj = bg['adj_list']
    x = feat_anon[cc_nodes].clone()

    node_set = set(cc_nodes)
    local_src, local_dst = [], []
    for g_src in cc_nodes:
        for g_dst in adj[g_src]:
            if g_dst in node_set and g_dst in local_map:
                local_src.append(local_map[g_src])
                local_dst.append(local_map[g_dst])

    if len(local_src) == 0:
        edge_index = torch.tensor([[0], [0]], dtype=torch.long)
    else:
        edge_index = torch.tensor([local_src, local_dst], dtype=torch.long)

    y = torch.tensor([meta['cc_label_map'][cc_id]], dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y)


def build_pyg_graph_ablation(cc_id, feature_mode='anonymous'):
    cc_nodes = sorted(bg['cc_to_nodes'][cc_id])
    local_map = {g: l for l, g in enumerate(cc_nodes)}
    adj = bg['adj_list']
    x_anon = feat_anon[cc_nodes].clone()

    node_set = set(cc_nodes)
    local_src, local_dst = [], []
    for g_src in cc_nodes:
        for g_dst in adj[g_src]:
            if g_dst in node_set and g_dst in local_map:
                local_src.append(local_map[g_src])
                local_dst.append(local_map[g_dst])

    if len(local_src) == 0:
        edge_index = torch.tensor([[0], [0]], dtype=torch.long)
    else:
        edge_index = torch.tensor([local_src, local_dst], dtype=torch.long)

    deg = degree(edge_index[1], x_anon.size(0), dtype=torch.float)
    deg = torch.log(deg + 1).view(-1, 1)

    if feature_mode == 'full':
        x = torch.cat([x_anon, deg], dim=1)
    elif feature_mode == 'anonymous':
        x = x_anon
    elif feature_mode == 'degree':
        x = deg
    else:
        raise ValueError(f'Unknown feature mode: {feature_mode}')

    y = torch.tensor([meta['cc_label_map'][cc_id]], dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y)

## 2.2 Model architectures

In [67]:
class GraphSentryGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN_DIM, dropout=DROPOUT):
        super().__init__()
        self.proj = torch.nn.Linear(in_channels, hidden)
        self.conv1 = GCNConv(hidden, hidden)
        self.bn1 = torch.nn.BatchNorm1d(hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.bn2 = torch.nn.BatchNorm1d(hidden)
        self.classifier = torch.nn.Linear(hidden, 2)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.proj(x))
        h = F.relu(self.bn1(self.conv1(x, edge_index)))
        h = self.bn2(self.conv2(h, edge_index)) + x
        h = global_max_pool(h, batch)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.classifier(h)


class GraphSentryGAT(torch.nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN_DIM, dropout=DROPOUT, heads=4):
        super().__init__()
        self.proj = torch.nn.Linear(in_channels, hidden)
        self.conv1 = GATConv(hidden, hidden // heads, heads=heads)
        self.bn1 = torch.nn.BatchNorm1d(hidden)
        self.conv2 = GATConv(hidden, hidden // heads, heads=heads)
        self.bn2 = torch.nn.BatchNorm1d(hidden)
        self.classifier = torch.nn.Linear(hidden, 2)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.proj(x))
        h = F.relu(self.bn1(self.conv1(x, edge_index)))
        h = self.bn2(self.conv2(h, edge_index)) + x
        h = global_max_pool(h, batch)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.classifier(h)


class GraphSentrySAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN_DIM, dropout=DROPOUT):
        super().__init__()
        self.proj = torch.nn.Linear(in_channels, hidden)
        self.conv1 = SAGEConv(hidden, hidden)
        self.bn1 = torch.nn.BatchNorm1d(hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.bn2 = torch.nn.BatchNorm1d(hidden)
        self.classifier = torch.nn.Linear(hidden, 2)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.proj(x))
        h = F.relu(self.bn1(self.conv1(x, edge_index)))
        h = self.bn2(self.conv2(h, edge_index)) + x
        h = global_max_pool(h, batch)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.classifier(h)


POOL_FN = {'mean': global_mean_pool, 'max': global_max_pool, 'add': global_add_pool}


class AblationGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN_DIM, n_layers=2,
                 pool='max', dropout=DROPOUT, use_residual=True):
        super().__init__()
        self.pool_fn = POOL_FN[pool]
        self.dropout = dropout
        self.use_residual = use_residual

        self.proj = torch.nn.Linear(in_channels, hidden)
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        for _ in range(n_layers):
            self.convs.append(GCNConv(hidden, hidden))
            self.bns.append(torch.nn.BatchNorm1d(hidden))
        self.classifier = torch.nn.Linear(hidden, 2)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.proj(x))
        for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):
            h = bn(conv(x, edge_index))
            if i < len(self.convs) - 1:
                h = F.relu(h)
            if self.use_residual:
                h = h + x
            x = h
        x = self.pool_fn(x, batch)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x)


MODEL_CLASSES = {'GCN': GraphSentryGCN, 'GAT': GraphSentryGAT}
BASELINE_CLASSES = [('GCN', GraphSentryGCN), ('GAT', GraphSentryGAT), ('SAGE', GraphSentrySAGE)]

INPUT_DIM = stats['feature_dims']['anonymous']
for name, cls in [*BASELINE_CLASSES]:
    print(f'{name:5s}: {sum(p.numel() for p in cls(INPUT_DIM).parameters()):,} params')

GCN  : 39,426 params
GAT  : 39,938 params
SAGE : 72,194 params


## 2.3 CC-aware GraphSAINT sampler

In [68]:
class SubgraphSAINTSampler:
    def __init__(self, eligible_cc_ids):
        self.eligible_ccs = set(eligible_cc_ids)
        self.N = bg['num_nodes']
        self.M = bg['num_edges']

        self.eligible_illicit = [c for c in self.eligible_ccs if meta['cc_label_map'][c] == 1]
        self.eligible_licit = [c for c in self.eligible_ccs if meta['cc_label_map'][c] == 0]
        self.cc_sizes = {cc: len(nodes) for cc, nodes in bg['cc_to_nodes'].items()}

    def sample(self, budget, strategy='node'):
        if strategy == 'node':
            sampled = set(random.sample(range(self.N), min(budget, self.N)))
        elif strategy == 'edge':
            ei = bg['edge_index']
            indices = random.sample(range(self.M), min(budget, self.M))
            sampled = set()
            for idx in indices:
                sampled.add(ei[0, idx].item())
                sampled.add(ei[1, idx].item())
        elif strategy == 'rw':
            adj = bg['adj_list']
            roots = random.sample(range(self.N), min(budget, self.N))
            sampled = set(roots)
            for root in roots:
                current = root
                for _ in range(5):
                    neighbors = adj[current]
                    if not neighbors:
                        break
                    current = random.choice(neighbors)
                    sampled.add(current)
        else:
            raise ValueError(f'Unknown strategy: {strategy}')

        node_to_cc = bg['node_to_cc']
        touched = {node_to_cc[n] for n in sampled if n in node_to_cc and node_to_cc[n] in self.eligible_ccs}

        if not touched:
            touched = set(random.sample(list(self.eligible_ccs), min(32, len(self.eligible_ccs))))

        touched_illicit = [c for c in touched if meta['cc_label_map'][c] == 1]
        touched_licit = [c for c in touched if meta['cc_label_map'][c] == 0]

        if touched_illicit and touched_licit:
            factor = max(1, len(touched_licit) // len(touched_illicit))
            cc_ids = touched_licit + touched_illicit * factor
        elif not touched_illicit:
            n_inject = max(1, len(touched_licit) // 10)
            injected = random.choices(self.eligible_illicit, k=min(n_inject, len(self.eligible_illicit)))
            cc_ids = touched_licit + injected
        else:
            cc_ids = list(touched)

        data_list = [build_pyg_graph(cc_id) for cc_id in cc_ids]
        norm_weights = self._compute_norms(cc_ids, budget, strategy)
        batch = Batch.from_data_list(data_list)
        return batch, norm_weights

    def _compute_norms(self, cc_ids, budget, strategy):
        weights = []
        for cc_id in cc_ids:
            sz = self.cc_sizes.get(cc_id, 1)
            if strategy == 'node':
                p = 1 - (1 - sz / self.N) ** budget
            elif strategy == 'edge':
                d_c = sum(len(bg['adj_list'][n]) for n in bg['cc_to_nodes'][cc_id])
                p = 1 - (1 - d_c / (2 * self.M + 1)) ** budget
            elif strategy == 'rw':
                p = 1 - (1 - sz / self.N) ** (budget * 5)
            else:
                p = 1.0
            weights.append(1.0 / max(p, 1e-6))
        w = torch.tensor(weights, dtype=torch.float)
        return w * len(w) / w.sum()


sampler = SubgraphSAINTSampler(meta['train_ids'])
print(f'Sampler ready: {len(sampler.eligible_ccs):,} train CCs '
      f'({len(sampler.eligible_illicit)} illicit, {len(sampler.eligible_licit)} licit)')

Sampler ready: 19,434 train CCs (1934 illicit, 17500 licit)


## 2.4 Loss functions + evaluation

In [69]:
def focal_loss(logits, targets, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA):
    ce = F.cross_entropy(logits, targets, reduction='none')
    pt = torch.exp(-ce)
    weights = alpha * (1 - pt) ** gamma
    return (weights * ce).mean()


def weighted_focal_loss(logits, targets, norm_weights, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA):
    ce = F.cross_entropy(logits, targets, reduction='none')
    pt = torch.exp(-ce)
    fl = alpha * (1 - pt) ** gamma * ce
    return (fl * norm_weights.to(logits.device)).mean()


def evaluate(model, loader):
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            prob = F.softmax(out, dim=1)[:, 1]
            y_true.extend(batch.y.cpu().numpy())
            y_prob.extend(prob.cpu().numpy())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    has_both = len(np.unique(y_true)) > 1
    return {
        'auroc': roc_auc_score(y_true, y_prob) if has_both else 0.0,
        'auc_pr': average_precision_score(y_true, y_prob) if has_both else 0.0,
        'f1': f1_score(y_true, (y_prob >= 0.5).astype(int), zero_division=0),
        'y_true': y_true,
        'y_prob': y_prob,
    }


def tune_threshold(y_true, y_prob):
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.1, 1.0, 0.05):
        f = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t, best_f1

## 2.5 Build val/test loaders

In [70]:
val_data = [build_pyg_graph(cc_id) for cc_id in meta['val_ids']]
test_data = [build_pyg_graph(cc_id) for cc_id in meta['test_ids']]
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)
print(f'Val: {len(val_data)}, Test: {len(test_data)}, dim = {val_data[0].x.size(1)}')

Val: 4164, Test: 4165, dim = 43


# Part 3: Model Training

Grid search across GCN/GAT backbones × node/edge/rw sampling strategies
using CC-aware GraphSAINT sampling with focal loss.

## 3.1 Grid search

In [71]:
all_results = {}
best_overall_auroc = 0
best_overall_state = None
best_overall_config = None


for backbone_name in BACKBONES:
    for strategy in STRATEGIES:
        config_name = f'{backbone_name}_{strategy}'
        print('=' * 70)
        print(f'{config_name}: {backbone_name} + GraphSAINT ({strategy} sampling)')
        print('=' * 70)

        torch.manual_seed(SEED)
        np.random.seed(SEED)
        random.seed(SEED)

        model_cls = MODEL_CLASSES[backbone_name]
        model = model_cls(INPUT_DIM).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE
        )

        best_val_auroc = 0
        best_model_state = None
        epochs_no_improve = 0
        history = []

        t_start = time.time()
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            epoch_loss = 0

            for _ in range(SAINT_SAMPLES_PER_EPOCH):
                batch, norm_weights = sampler.sample(SAINT_BUDGET, strategy)
                batch = batch.to(device)

                optimizer.zero_grad()
                out = model(batch.x, batch.edge_index, batch.batch)
                loss = weighted_focal_loss(out, batch.y, norm_weights)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()

            avg_loss = epoch_loss / SAINT_SAMPLES_PER_EPOCH
            val_m = evaluate(model, val_loader)
            scheduler.step(val_m['auroc'])

            if val_m['auroc'] > best_val_auroc:
                best_val_auroc = val_m['auroc']
                best_model_state = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0
                marker = ' *'
            else:
                epochs_no_improve += 1
                marker = ''

            lr_now = optimizer.param_groups[0]['lr']
            history.append({'epoch': epoch, 'loss': avg_loss,
                            'val_auroc': val_m['auroc'], 'val_f1': val_m['f1'], 'lr': lr_now})

            if epoch <= 5 or epoch % 5 == 0 or marker:
                print(f'  Epoch {epoch:3d} | Loss: {avg_loss:.4f} | '
                      f"Val AUROC: {val_m['auroc']:.4f} | Val F1: {val_m['f1']:.4f} | "
                      f'LR: {lr_now:.6f}{marker}')

            if epochs_no_improve >= PATIENCE:
                print(f'\n  Early stopping at epoch {epoch}')
                break

        t_elapsed = time.time() - t_start
        model.load_state_dict(best_model_state)

        val_r = evaluate(model, val_loader)
        best_thresh, val_f1_tuned = tune_threshold(val_r['y_true'], val_r['y_prob'])

        test_r = evaluate(model, test_loader)
        y_pred = (test_r['y_prob'] >= best_thresh).astype(int)
        cm = confusion_matrix(test_r['y_true'], y_pred)

        result = {
            'backbone': backbone_name,
            'strategy': strategy,
            'best_val_auroc': round(best_val_auroc, 4),
            'threshold': best_thresh,
            'test_auroc': round(test_r['auroc'], 4),
            'test_auc_pr': round(test_r['auc_pr'], 4),
            'test_f1': round(f1_score(test_r['y_true'], y_pred, zero_division=0), 4),
            'test_precision': round(precision_score(test_r['y_true'], y_pred, zero_division=0), 4),
            'test_recall': round(recall_score(test_r['y_true'], y_pred, zero_division=0), 4),
            'confusion_matrix': {'tn': int(cm[0,0]), 'fp': int(cm[0,1]),
                                 'fn': int(cm[1,0]), 'tp': int(cm[1,1])},
            'epochs_run': len(history),
            'training_time_s': round(t_elapsed, 1),
        }

        all_results[config_name] = result

        if result['test_auroc'] > best_overall_auroc:
            best_overall_auroc = result['test_auroc']
            best_overall_state = copy.deepcopy(best_model_state)
            best_overall_config = config_name


        print(f"  => Test AUROC: {result['test_auroc']:.4f} | "
              f"AUC-PR: {result['test_auc_pr']:.4f} | "
              f"F1: {result['test_f1']:.4f} @ thresh={best_thresh:.2f} | "
              f'Time: {t_elapsed:.1f}s\n')

GCN_node: GCN + GraphSAINT (node sampling)
  Epoch   1 | Loss: 0.1960 | Val AUROC: 0.8554 | Val F1: 0.3560 | LR: 0.005000 *
  Epoch   2 | Loss: 0.1030 | Val AUROC: 0.8719 | Val F1: 0.4169 | LR: 0.005000 *
  Epoch   3 | Loss: 0.0893 | Val AUROC: 0.8601 | Val F1: 0.4066 | LR: 0.005000
  Epoch   4 | Loss: 0.0915 | Val AUROC: 0.8788 | Val F1: 0.4349 | LR: 0.005000 *
  Epoch   5 | Loss: 0.0799 | Val AUROC: 0.8650 | Val F1: 0.4213 | LR: 0.005000
  Epoch   6 | Loss: 0.0815 | Val AUROC: 0.8908 | Val F1: 0.4259 | LR: 0.005000 *
  Epoch   9 | Loss: 0.0782 | Val AUROC: 0.8971 | Val F1: 0.4870 | LR: 0.005000 *
  Epoch  10 | Loss: 0.0700 | Val AUROC: 0.8822 | Val F1: 0.4510 | LR: 0.005000
  Epoch  13 | Loss: 0.0716 | Val AUROC: 0.8996 | Val F1: 0.4394 | LR: 0.005000 *
  Epoch  15 | Loss: 0.0729 | Val AUROC: 0.8885 | Val F1: 0.4515 | LR: 0.005000
  Epoch  16 | Loss: 0.0725 | Val AUROC: 0.9037 | Val F1: 0.4916 | LR: 0.005000 *
  Epoch  19 | Loss: 0.0702 | Val AUROC: 0.9074 | Val F1: 0.4744 | LR: 0.00

## 3.2 Results summary

In [72]:
print('=' * 90)
print('GRAPHSENTRY MODEL SEARCH RESULTS')
print('=' * 90)
print(f'{"Config":<20s} {"AUROC":>8s} {"AUC-PR":>8s} {"F1":>8s} '
      f'{"Prec":>8s} {"Recall":>8s} {"Thresh":>8s} {"Time":>8s}')
print('-' * 90)
for name, r in sorted(all_results.items(), key=lambda x: -x[1]['test_auroc']):
    tag = ' <-- BEST' if name == best_overall_config else ''
    print(f"{name:<20s} {r['test_auroc']:>8.4f} {r['test_auc_pr']:>8.4f} "
          f"{r['test_f1']:>8.4f} {r['test_precision']:>8.4f} {r['test_recall']:>8.4f} "
          f"{r['threshold']:>8.2f} {r['training_time_s']:>7.1f}s{tag}")
print('-' * 90)

GRAPHSENTRY MODEL SEARCH RESULTS
Config                  AUROC   AUC-PR       F1     Prec   Recall   Thresh     Time
------------------------------------------------------------------------------------------
GAT_rw                 0.9348   0.7228   0.6279   0.5854   0.6771     0.65    76.6s <-- BEST
GAT_edge               0.9339   0.7276   0.6399   0.6675   0.6145     0.65   112.2s
GAT_node               0.9319   0.7158   0.6045   0.5720   0.6410     0.65    77.1s
GCN_node               0.9316   0.7135   0.6217   0.5598   0.6988     0.60    98.8s
GCN_edge               0.9310   0.7092   0.6165   0.6313   0.6024     0.65   107.1s
GCN_rw                 0.9304   0.7159   0.6332   0.6145   0.6530     0.65    91.9s
------------------------------------------------------------------------------------------


## 3.3 Save model + results

In [73]:
torch.save(best_overall_state, os.path.join(PROCESSED_PATH, 'model_a.pth'))

training_results = {
    'architecture': '2-layer with residual, max pool, focal loss',
    'input_dim': INPUT_DIM,
    'features': 'anonymous_43dim',
    'best_config': best_overall_config,
    'all_configs': all_results,
}
with open(os.path.join(PROCESSED_PATH, 'training_results.json'), 'w') as f:
    json.dump(training_results, f, indent=2)

print(f'Best: {best_overall_config} (AUROC: {best_overall_auroc:.4f})')
print(f"Saved model_a.pth ({os.path.getsize(os.path.join(PROCESSED_PATH, 'model_a.pth'))/1024:.1f} KB)")

Best: GAT_rw (AUROC: 0.9348)
Saved model_a.pth (164.8 KB)


# Part 4: Baselines

Same architecture as GraphSentry but using standard DataLoader (no GraphSAINT).
Isolates the sampling contribution.

## 4.1 Build oversampled train loader

In [74]:
train_data = [build_pyg_graph(cc_id) for cc_id in meta['train_ids']]
train_pos = [d for d in train_data if d.y.item() == 1]
train_neg = [d for d in train_data if d.y.item() == 0]
train_balanced = train_neg + train_pos * max(1, len(train_neg) // len(train_pos))
train_loader = DataLoader(train_balanced, batch_size=BATCH_SIZE, shuffle=True)
print(f'Train: {len(train_balanced)} (oversampled from {len(train_data)})')

Train: 34906 (oversampled from 19434)


## 4.2 Training function

In [75]:
def train_and_evaluate_baseline(model_cls, name):
    torch.manual_seed(SEED)
    model = model_cls(INPUT_DIM).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE)

    best_val_auroc, best_state, no_improve = 0, None, 0
    t0 = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = focal_loss(out, batch.y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        val_m = evaluate(model, val_loader)
        scheduler.step(val_m['auroc'])

        if val_m['auroc'] > best_val_auroc:
            best_val_auroc = val_m['auroc']
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
            marker = ' *'
        else:
            no_improve += 1
            marker = ''

        if epoch <= 5 or epoch % 5 == 0 or marker:
            print(f"  {name:5s} Epoch {epoch:3d} | Loss: {avg_loss:.4f} | "
                  f"Val AUROC: {val_m['auroc']:.4f} | Val F1: {val_m['f1']:.4f}{marker}")

        if no_improve >= PATIENCE:
            print(f'  {name:5s} Early stopping at epoch {epoch}')
            break

    t_elapsed = time.time() - t0
    model.load_state_dict(best_state)

    val_r = evaluate(model, val_loader)
    thresh, _ = tune_threshold(val_r['y_true'], val_r['y_prob'])
    test_r = evaluate(model, test_loader)
    y_pred = (test_r['y_prob'] >= thresh).astype(int)
    cm = confusion_matrix(test_r['y_true'], y_pred)

    result = {
        'name': name, 'threshold': thresh,
        'test_auroc': round(test_r['auroc'], 4),
        'test_auc_pr': round(test_r['auc_pr'], 4),
        'test_f1': round(f1_score(test_r['y_true'], y_pred, zero_division=0), 4),
        'test_precision': round(precision_score(test_r['y_true'], y_pred, zero_division=0), 4),
        'test_recall': round(recall_score(test_r['y_true'], y_pred, zero_division=0), 4),
        'confusion_matrix': {'tn': int(cm[0,0]), 'fp': int(cm[0,1]),
                             'fn': int(cm[1,0]), 'tp': int(cm[1,1])},
        'epochs_run': epoch, 'training_time_s': round(t_elapsed, 1),
    }
    print(f"  {name:5s} => AUROC: {result['test_auroc']:.4f} | "
          f"AUC-PR: {result['test_auc_pr']:.4f} | "
          f"F1: {result['test_f1']:.4f} @ {thresh:.2f} | {t_elapsed:.1f}s\n")
    return result

## 4.3 Run baselines

In [76]:
baseline_results = {}
for name, cls in BASELINE_CLASSES:
    print('=' * 70)
    print(f'Baseline: {name} (DataLoader, no GraphSAINT)')
    print('=' * 70)
    baseline_results[name.lower()] = train_and_evaluate_baseline(cls, name)

Baseline: GCN (DataLoader, no GraphSAINT)
  GCN   Epoch   1 | Loss: 0.0882 | Val AUROC: 0.8863 | Val F1: 0.4139 *
  GCN   Epoch   2 | Loss: 0.0718 | Val AUROC: 0.9064 | Val F1: 0.4974 *
  GCN   Epoch   3 | Loss: 0.0652 | Val AUROC: 0.9099 | Val F1: 0.5225 *
  GCN   Epoch   4 | Loss: 0.0612 | Val AUROC: 0.9033 | Val F1: 0.4614
  GCN   Epoch   5 | Loss: 0.0578 | Val AUROC: 0.9173 | Val F1: 0.5087 *
  GCN   Epoch   6 | Loss: 0.0559 | Val AUROC: 0.9190 | Val F1: 0.4876 *
  GCN   Epoch  10 | Loss: 0.0455 | Val AUROC: 0.9155 | Val F1: 0.5228
  GCN   Epoch  15 | Loss: 0.0272 | Val AUROC: 0.9127 | Val F1: 0.5944
  GCN   Early stopping at epoch 16
  GCN   => AUROC: 0.9310 | AUC-PR: 0.7211 | F1: 0.6301 @ 0.70 | 68.8s

Baseline: GAT (DataLoader, no GraphSAINT)
  GAT   Epoch   1 | Loss: 0.0859 | Val AUROC: 0.9098 | Val F1: 0.4743 *
  GAT   Epoch   2 | Loss: 0.0675 | Val AUROC: 0.9136 | Val F1: 0.5159 *
  GAT   Epoch   3 | Loss: 0.0653 | Val AUROC: 0.9181 | Val F1: 0.5004 *
  GAT   Epoch   4 | Loss

## 4.4 Comparison + save

In [77]:
best_cfg = best_overall_config
gs_r = all_results[best_cfg]

print('=' * 90)
print('COMPARISON: GraphSentry vs Baselines')
print('=' * 90)
print(f'{"Model":<35s} {"AUROC":>8s} {"AUC-PR":>8s} {"F1":>8s} {"Prec":>8s} {"Recall":>8s}')
print('-' * 90)
print(f'{"GraphSentry (" + best_cfg + ")":<35s} '
      f"{gs_r['test_auroc']:>8.4f} {gs_r['test_auc_pr']:>8.4f} "
      f"{gs_r['test_f1']:>8.4f} {gs_r['test_precision']:>8.4f} {gs_r['test_recall']:>8.4f}")
for key in ['gcn', 'gat', 'sage']:
    r = baseline_results[key]
    print(f"{r['name'] + ' (DataLoader)':<35s} "
          f"{r['test_auroc']:>8.4f} {r['test_auc_pr']:>8.4f} "
          f"{r['test_f1']:>8.4f} {r['test_precision']:>8.4f} {r['test_recall']:>8.4f}")
print('-' * 90)

with open(os.path.join(PROCESSED_PATH, 'baseline_results.json'), 'w') as f:
    json.dump(baseline_results, f, indent=2)
print('\nSaved baseline_results.json')

COMPARISON: GraphSentry vs Baselines
Model                                  AUROC   AUC-PR       F1     Prec   Recall
------------------------------------------------------------------------------------------
GraphSentry (GAT_rw)                  0.9348   0.7228   0.6279   0.5854   0.6771
GCN (DataLoader)                      0.9310   0.7211   0.6301   0.6241   0.6361
GAT (DataLoader)                      0.9230   0.6969   0.5983   0.5287   0.6892
SAGE (DataLoader)                     0.9295   0.7008   0.6203   0.6394   0.6024
------------------------------------------------------------------------------------------

Saved baseline_results.json


# Part 5: Ablation Studies

Control = optimised config (43-dim anon, 2-layer, max pool, residual, focal loss).
Each ablation disables one component while holding others constant.

## 5.1 Ablation data loaders

In [78]:
def build_ablation_loaders(feature_mode='anonymous'):
    data = [build_pyg_graph_ablation(cc_id, feature_mode) for cc_id in meta['train_ids']]
    val = [build_pyg_graph_ablation(cc_id, feature_mode) for cc_id in meta['val_ids']]
    test = [build_pyg_graph_ablation(cc_id, feature_mode) for cc_id in meta['test_ids']]

    pos = [d for d in data if d.y.item() == 1]
    neg = [d for d in data if d.y.item() == 0]
    balanced = neg + pos * max(1, len(neg) // len(pos))

    return (
        DataLoader(balanced, batch_size=BATCH_SIZE, shuffle=True),
        DataLoader(val, batch_size=BATCH_SIZE),
        DataLoader(test, batch_size=BATCH_SIZE),
        data[0].x.size(1),
    )

## 5.2 Ablation training function

In [79]:
def run_ablation(name, feature_mode='anonymous', pool='max', n_layers=2,
                 use_residual=True, use_focal=True):
    torch.manual_seed(SEED)
    train_loader, val_loader, test_loader, input_dim = build_ablation_loaders(feature_mode)

    model = AblationGCN(input_dim, n_layers=n_layers, pool=pool,
                        use_residual=use_residual).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE)

    best_val_auroc, best_state, no_improve = 0, None, 0
    t0 = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = focal_loss(out, batch.y) if use_focal else F.cross_entropy(out, batch.y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        val_m = evaluate(model, val_loader)
        scheduler.step(val_m['auroc'])

        if val_m['auroc'] > best_val_auroc:
            best_val_auroc = val_m['auroc']
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
            marker = ' *'
        else:
            no_improve += 1
            marker = ''

        if epoch <= 3 or epoch % 5 == 0 or marker:
            print(f"  {name[:6]:6s} Epoch {epoch:3d} | Loss: {avg_loss:.4f} | "
                  f"Val AUROC: {val_m['auroc']:.4f} | Val F1: {val_m['f1']:.4f}{marker}")

        if no_improve >= PATIENCE:
            print(f'  {name[:6]:6s} Early stopping at epoch {epoch}')
            break

    t_elapsed = time.time() - t0
    model.load_state_dict(best_state)

    val_r = evaluate(model, val_loader)
    best_thresh, _ = tune_threshold(val_r['y_true'], val_r['y_prob'])
    test_r = evaluate(model, test_loader)
    y_pred = (test_r['y_prob'] >= best_thresh).astype(int)
    cm = confusion_matrix(test_r['y_true'], y_pred)

    result = {
        'name': name, 'input_dim': input_dim,
        'test_auroc': round(test_r['auroc'], 4),
        'test_auc_pr': round(test_r['auc_pr'], 4),
        'test_f1': round(f1_score(test_r['y_true'], y_pred, zero_division=0), 4),
        'test_precision': round(precision_score(test_r['y_true'], y_pred, zero_division=0), 4),
        'test_recall': round(recall_score(test_r['y_true'], y_pred, zero_division=0), 4),
        'threshold': best_thresh,
        'training_time_s': round(t_elapsed, 1),
    }
    print(f"  {name} => AUROC: {result['test_auroc']:.4f} | AUC-PR: {result['test_auc_pr']:.4f} | "
          f"F1: {result['test_f1']:.4f} @ {best_thresh:.2f} | Time: {t_elapsed:.1f}s\n")
    return result

## 5.3 Run ablations

In [80]:
ablation_results = {}

print('=' * 70)
print('CONTROL: 43-dim anon, 2-layer, max pool, residual, focal loss')
print('=' * 70)
ablation_results['control'] = run_ablation('Control')

print('=' * 70)
print('A1: Full features (44-dim = anon + degree)')
print('=' * 70)
ablation_results['a1_full'] = run_ablation('A1-Full', feature_mode='full')

print('=' * 70)
print('A2: Degree only (1-dim)')
print('=' * 70)
ablation_results['a2_degree'] = run_ablation('A2-Deg', feature_mode='degree')

print('=' * 70)
print('B1: Mean pool')
print('=' * 70)
ablation_results['b1_mean'] = run_ablation('B1-Mean', pool='mean')

print('=' * 70)
print('B2: Add pool')
print('=' * 70)
ablation_results['b2_add'] = run_ablation('B2-Add', pool='add')

print('=' * 70)
print('C1: 3 layers')
print('=' * 70)
ablation_results['c1_3layer'] = run_ablation('C1-3L', n_layers=3)

print('=' * 70)
print('C2: 4 layers')
print('=' * 70)
ablation_results['c2_4layer'] = run_ablation('C2-4L', n_layers=4)

print('=' * 70)
print('D1: Cross-entropy (no focal)')
print('=' * 70)
ablation_results['d1_ce'] = run_ablation('D1-CE', use_focal=False)

print('=' * 70)
print('E1: No residual connections')
print('=' * 70)
ablation_results['e1_nores'] = run_ablation('E1-NoRes', use_residual=False)

CONTROL: 43-dim anon, 2-layer, max pool, residual, focal loss
  Contro Epoch   1 | Loss: 0.0936 | Val AUROC: 0.8762 | Val F1: 0.4240 *
  Contro Epoch   2 | Loss: 0.0732 | Val AUROC: 0.9029 | Val F1: 0.4661 *
  Contro Epoch   3 | Loss: 0.0668 | Val AUROC: 0.9049 | Val F1: 0.5230 *
  Contro Epoch   4 | Loss: 0.0622 | Val AUROC: 0.9064 | Val F1: 0.4552 *
  Contro Epoch   5 | Loss: 0.0590 | Val AUROC: 0.9163 | Val F1: 0.5284 *
  Contro Epoch   7 | Loss: 0.0539 | Val AUROC: 0.9168 | Val F1: 0.4918 *
  Contro Epoch   8 | Loss: 0.0508 | Val AUROC: 0.9194 | Val F1: 0.5443 *
  Contro Epoch  10 | Loss: 0.0463 | Val AUROC: 0.9165 | Val F1: 0.5266
  Contro Epoch  13 | Loss: 0.0421 | Val AUROC: 0.9200 | Val F1: 0.5593 *
  Contro Epoch  15 | Loss: 0.0378 | Val AUROC: 0.9154 | Val F1: 0.5531
  Contro Epoch  20 | Loss: 0.0231 | Val AUROC: 0.9101 | Val F1: 0.5773
  Contro Early stopping at epoch 23
  Control => AUROC: 0.9326 | AUC-PR: 0.7215 | F1: 0.6225 @ 0.65 | Time: 100.0s

A1: Full features (44-dim

## 5.4 Ablation summary + save

In [81]:
ctrl = ablation_results['control']

print('=' * 90)
print('ABLATION RESULTS')
print('=' * 90)
print(f'{"Experiment":<25s} {"Dim":>5s} {"AUROC":>8s} {"AUC-PR":>8s} {"F1":>8s} '
      f'{"Prec":>8s} {"Recall":>8s} {"dAUROC":>8s}')
print('-' * 90)

for key in ['control', 'a1_full', 'a2_degree', 'b1_mean', 'b2_add',
            'c1_3layer', 'c2_4layer', 'd1_ce', 'e1_nores']:
    r = ablation_results[key]
    delta = r['test_auroc'] - ctrl['test_auroc']
    print(f"{r['name']:<25s} {r['input_dim']:>5d} {r['test_auroc']:>8.4f} "
          f"{r['test_auc_pr']:>8.4f} {r['test_f1']:>8.4f} "
          f"{r['test_precision']:>8.4f} {r['test_recall']:>8.4f} {delta:>+8.4f}")
print('-' * 90)

# Cross-notebook sampling ablation

print(f'\nCross-notebook sampling ablation:')
print(f'  GraphSentry ({best_overall_config}): AUROC {all_results[best_overall_config]["test_auroc"]}')
print(f'  GCN DataLoader: AUROC {baseline_results["gcn"]["test_auroc"]}')

with open(os.path.join(PROCESSED_PATH, 'ablation_results.json'), 'w') as f:
    json.dump(ablation_results, f, indent=2)
print('Saved ablation_results.json')

ABLATION RESULTS
Experiment                  Dim    AUROC   AUC-PR       F1     Prec   Recall   dAUROC
------------------------------------------------------------------------------------------
Control                      43   0.9326   0.7215   0.6225   0.6468   0.6000  +0.0000
A1-Full                      44   0.9254   0.6980   0.6007   0.5690   0.6361  -0.0072
A2-Deg                        1   0.5164   0.1061   0.1800   0.0995   0.9373  -0.4162
B1-Mean                      43   0.9263   0.6880   0.5870   0.5660   0.6096  -0.0063
B2-Add                       43   0.9289   0.7215   0.6168   0.5986   0.6361  -0.0037
C1-3L                        43   0.9266   0.7216   0.6147   0.5645   0.6747  -0.0060
C2-4L                        43   0.9315   0.7144   0.6368   0.6399   0.6337  -0.0011
D1-CE                        43   0.9357   0.7267   0.6275   0.6124   0.6434  +0.0031
E1-NoRes                     43   0.9304   0.7222   0.6307   0.6277   0.6337  -0.0022
--------------------------------

# Part 6: Statistical Significance

Runs the best GraphSentry config and all baselines across 5 random seeds.
Paired t-tests + Cohen's d effect size.

## 6.1 Read best config

In [82]:
BEST_BACKBONE = best_overall_config.split('_')[0]
BEST_STRATEGY = best_overall_config.split('_')[1]
print(f'Best config: {best_overall_config} (backbone={BEST_BACKBONE}, strategy={BEST_STRATEGY})')

Best config: GAT_rw (backbone=GAT, strategy=rw)


## 6.2 Seed-paired training functions

In [83]:
def run_saint(seed, backbone_name, strategy):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = MODEL_CLASSES[backbone_name](INPUT_DIM).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE)

    best_auroc, best_st, no_imp = 0, None, 0
    t0 = time.time()

    for ep in range(1, MAX_EPOCHS + 1):
        model.train()
        for _ in range(SAINT_SAMPLES_PER_EPOCH):
            batch, _ = sampler.sample(SAINT_BUDGET, strategy)
            batch = batch.to(device)
            opt.zero_grad()
            focal_loss(model(batch.x, batch.edge_index, batch.batch), batch.y).backward()
            opt.step()

        vm = evaluate(model, val_loader)
        sch.step(vm['auroc'])

        if vm['auroc'] > best_auroc:
            best_auroc = vm['auroc']
            best_st = copy.deepcopy(model.state_dict())
            no_imp = 0
        else:
            no_imp += 1

        if no_imp >= PATIENCE:
            break

    model.load_state_dict(best_st)
    tm = evaluate(model, test_loader)
    return tm['auroc'], tm['auc_pr'], tm['f1'], time.time() - t0


def run_baseline(seed, model_cls, name):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = model_cls(INPUT_DIM).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE)

    best_auroc, best_st, no_imp = 0, None, 0
    t0 = time.time()

    for ep in range(1, MAX_EPOCHS + 1):
        model.train()
        for b in train_loader:
            b = b.to(device)
            opt.zero_grad()
            focal_loss(model(b.x, b.edge_index, b.batch), b.y).backward()
            opt.step()

        vm = evaluate(model, val_loader)
        sch.step(vm['auroc'])

        if vm['auroc'] > best_auroc:
            best_auroc = vm['auroc']
            best_st = copy.deepcopy(model.state_dict())
            no_imp = 0
        else:
            no_imp += 1

        if no_imp >= PATIENCE:
            break

    model.load_state_dict(best_st)
    tm = evaluate(model, test_loader)
    return tm['auroc'], tm['auc_pr'], tm['f1'], time.time() - t0


## 6.3 Run all seeds

In [84]:
all_runs = {}

for seed in SEEDS:
    print('=' * 70)
    print(f'Seed {seed} ({SEEDS.index(seed)+1}/{len(SEEDS)})')
    print('=' * 70)

    a, ap, f, t = run_saint(seed, BEST_BACKBONE, BEST_STRATEGY)
    all_runs.setdefault('GraphSentry', []).append({'seed': seed, 'auroc': a, 'auc_pr': ap, 'f1': f})
    print(f'  GraphSentry ({BEST_BACKBONE}+{BEST_STRATEGY})... AUROC={a:.4f}, F1={f:.4f}, {t:.1f}s')

    for bname, bcls in BASELINE_CLASSES:
        a, ap, f, t = run_baseline(seed, bcls, bname)
        all_runs.setdefault(bname, []).append({'seed': seed, 'auroc': a, 'auc_pr': ap, 'f1': f})
        print(f'  {bname} (DataLoader)... AUROC={a:.4f}, F1={f:.4f}, {t:.1f}s')

print('\nAll runs complete.')

Seed 42 (1/5)
  GraphSentry (GAT+rw)... AUROC=0.9302, F1=0.4997, 52.8s
  GCN (DataLoader)... AUROC=0.9236, F1=0.5688, 98.7s
  GAT (DataLoader)... AUROC=0.9266, F1=0.5591, 89.2s
  SAGE (DataLoader)... AUROC=0.9259, F1=0.5280, 50.7s
Seed 123 (2/5)
  GraphSentry (GAT+rw)... AUROC=0.9339, F1=0.5426, 74.2s
  GCN (DataLoader)... AUROC=0.9266, F1=0.5759, 107.2s
  GAT (DataLoader)... AUROC=0.9277, F1=0.5904, 116.4s
  SAGE (DataLoader)... AUROC=0.9284, F1=0.5416, 62.9s
Seed 456 (3/5)
  GraphSentry (GAT+rw)... AUROC=0.9308, F1=0.5594, 103.7s
  GCN (DataLoader)... AUROC=0.9292, F1=0.5871, 94.0s
  GAT (DataLoader)... AUROC=0.9188, F1=0.5813, 109.7s
  SAGE (DataLoader)... AUROC=0.9296, F1=0.5194, 55.8s
Seed 789 (4/5)
  GraphSentry (GAT+rw)... AUROC=0.9275, F1=0.5343, 73.8s
  GCN (DataLoader)... AUROC=0.9298, F1=0.5344, 64.4s
  GAT (DataLoader)... AUROC=0.9309, F1=0.5287, 83.7s
  SAGE (DataLoader)... AUROC=0.9239, F1=0.5435, 70.9s
Seed 1024 (5/5)
  GraphSentry (GAT+rw)... AUROC=0.9316, F1=0.5671, 79

## 6.4 Summary + significance tests + save

In [86]:
summary = {}
for model_name, runs in all_runs.items():
    aurocs = [r['auroc'] for r in runs]
    f1s = [r['f1'] for r in runs]
    auc_prs = [r['auc_pr'] for r in runs]
    summary[model_name] = {
        'auroc_mean': round(np.mean(aurocs), 4), 'auroc_std': round(np.std(aurocs), 4),
        'auc_pr_mean': round(np.mean(auc_prs), 4), 'auc_pr_std': round(np.std(auc_prs), 4),
        'f1_mean': round(np.mean(f1s), 4), 'f1_std': round(np.std(f1s), 4),
        'aurocs': aurocs, 'f1s': f1s,
    }

print('=' * 80)
print('SUMMARY: Mean +/- Std across', len(SEEDS), 'seeds')
print('=' * 80)
print(f'{"Model":<25s} {"AUROC":>15s} {"AUC-PR":>15s} {"F1":>15s}')
print('-' * 80)
for name, s in summary.items():
    print(f"{name:<25s} {s['auroc_mean']:.4f} +/- {s['auroc_std']:.4f} "
          f"{s['auc_pr_mean']:.4f} +/- {s['auc_pr_std']:.4f} "
          f"{s['f1_mean']:.4f} +/- {s['f1_std']:.4f}")
print('-' * 80)

gs_aurocs = summary['GraphSentry']['aurocs']
sig_results = []
print('\n' + '=' * 80)
print('STATISTICAL SIGNIFICANCE: GraphSentry vs each baseline')
print('=' * 80)
print(f'{"Comparison":<30s} {"Mean d":>8s} {"t-stat":>8s} {"p":>10s} {"Cohen d":>9s} {"Sig?":>5s}')
print('-' * 80)

for bname in [n for n, _ in BASELINE_CLASSES]:
    bl_aurocs = summary[bname]['aurocs']
    t_stat, p_val = scipy_stats.ttest_rel(gs_aurocs, bl_aurocs)
    diff = np.array(gs_aurocs) - np.array(bl_aurocs)
    pooled = np.sqrt((np.std(gs_aurocs)**2 + np.std(bl_aurocs)**2) / 2)
    cohen_d = np.mean(diff) / pooled if pooled > 0 else 0
    sig = 'Yes' if p_val < 0.05 else 'No'
    print(f'  GS vs {bname:<22s} {np.mean(diff):>+8.4f} {t_stat:>8.3f} {p_val:>10.6f} {cohen_d:>9.3f} {sig:>5s}')
    sig_results.append({'comparison': f'GS vs {bname}', 't_stat': round(t_stat, 3),
                         'p_value': round(p_val, 6), 'cohen_d': round(cohen_d, 3), 'significant': sig == 'Yes'})

print('-' * 80)

output = {
    'seeds': SEEDS,
    'best_config': f'{BEST_BACKBONE}_{BEST_STRATEGY}',
    'per_seed_runs': all_runs,
    'summary': {k: {sk: sv for sk, sv in v.items() if sk not in ('aurocs', 'f1s')}
                for k, v in summary.items()},
    'significance_tests': sig_results,
}

with open(os.path.join(PROCESSED_PATH, 'significance_results.json'), 'w') as f:
    json.dump(output, f, indent=2, default=str)

print('Saved significance_results.json')
print('\nNotebook sequence complete.')

SUMMARY: Mean +/- Std across 5 seeds
Model                               AUROC          AUC-PR              F1
--------------------------------------------------------------------------------
GraphSentry               0.9308 +/- 0.0021 0.7111 +/- 0.0115 0.5406 +/- 0.0236
GCN                       0.9275 +/- 0.0022 0.7111 +/- 0.0070 0.5634 +/- 0.0187
GAT                       0.9256 +/- 0.0040 0.7012 +/- 0.0114 0.5621 +/- 0.0220
SAGE                      0.9272 +/- 0.0020 0.7044 +/- 0.0074 0.5361 +/- 0.0106
--------------------------------------------------------------------------------

STATISTICAL SIGNIFICANCE: GraphSentry vs each baseline
Comparison                       Mean d   t-stat          p   Cohen d  Sig?
--------------------------------------------------------------------------------
  GS vs GCN                     +0.0033    1.882   0.133047     1.525    No
  GS vs GAT                     +0.0051    2.027   0.112552     1.601    No
  GS vs SAGE                    +0.0036   

# Part 7: Full-Scale Training

Rebuilds the entire data pipeline from raw Elliptic2 CSVs at full scale
(121K CCs, 444K nodes). Fully self-contained — does not depend on Part 1 artefacts.

## 7.1 Full-scale configuration

In [87]:
FEATURE_CHUNK_SIZE = 500_000
SAINT_STRATEGY = BEST_STRATEGY
SAINT_BUDGET = 1000
print(f'Full-scale: {BEST_BACKBONE} + {BEST_STRATEGY}, budget={SAINT_BUDGET}')


Full-scale: GAT + rw, budget=1000


## 7.2 Load raw data

In [88]:
df_labels = pd.read_csv(f'{RAW_PATH}/connected_components.csv')
df_labels['label'] = (df_labels['ccLabel'] != 'licit').astype(int)
df_nodes_raw = pd.read_csv(f'{RAW_PATH}/nodes.csv')
df_edges_raw = pd.read_csv(f'{RAW_PATH}/edges.csv')

cc_sizes_df = df_nodes_raw.groupby('ccId').size().reset_index(name='size')
df_labels_with_size = df_labels.merge(cc_sizes_df, on='ccId', how='left')
df_labels_with_size = df_labels_with_size[df_labels_with_size['size'] >= MIN_CC_SIZE]
selected_ccs = df_labels_with_size.copy()

target_cc_ids = set(selected_ccs['ccId'].values)
cc_label_map = dict(zip(selected_ccs['ccId'], selected_ccs['label']))

n_ill = int(selected_ccs['label'].sum())
n_lic = len(selected_ccs) - n_ill
print(f'All CCs: {len(selected_ccs):,} ({n_ill:,} illicit, {n_lic:,} licit)')

All CCs: 121,810 (2,763 illicit, 119,047 licit)


## 7.3 Build background graph

In [89]:
df_nodes = df_nodes_raw[df_nodes_raw['ccId'].isin(target_cc_ids)].copy()
df_nodes = df_nodes.merge(selected_ccs[['ccId', 'label']], on='ccId')

unique_nodes = np.sort(df_nodes['clId'].unique())
node_to_idx = {int(nid): i for i, nid in enumerate(unique_nodes)}
N = len(unique_nodes)

node_to_cc, cc_to_nodes = {}, {}
for cc_id, group in df_nodes.groupby('ccId'):
    idxs = [node_to_idx[n] for n in group['clId'].values]
    cc_to_nodes[cc_id] = sorted(idxs)
    for i in idxs: node_to_cc[i] = cc_id

node_id_set = set(unique_nodes)
em = df_edges_raw['clId1'].isin(node_id_set) & df_edges_raw['clId2'].isin(node_id_set)
df_edges = df_edges_raw[em]
src = np.array([node_to_idx[n] for n in df_edges['clId1'].values])
dst = np.array([node_to_idx[n] for n in df_edges['clId2'].values])
edge_index = torch.tensor(np.stack([src, dst]), dtype=torch.long)

adj_list = [[] for _ in range(N)]
for s, d in zip(src, dst): adj_list[s].append(d); adj_list[d].append(s)

bg = {'edge_index': edge_index, 'adj_list': adj_list, 'node_to_idx': node_to_idx,
      'node_to_cc': node_to_cc, 'cc_to_nodes': cc_to_nodes, 'num_nodes': N, 'num_edges': edge_index.size(1)}

print(f'Nodes: {N:,}, Edges: {edge_index.size(1):,}, CCs: {len(cc_to_nodes):,}')
print(f'RAM used: {psutil.Process().memory_info().rss / 1e9:.1f} GB')

Nodes: 444,521, Edges: 367,137, CCs: 121,810
RAM used: 2.7 GB


## 7.4 Feature extraction + split

In [90]:
feat = torch.zeros((N, 43), dtype=torch.float)
for chunk in tqdm(pd.read_csv(f'{RAW_PATH}/background_nodes.csv', chunksize=FEATURE_CHUNK_SIZE), desc='Features'):
    m = chunk['clId'].isin(node_id_set)
    matched = chunk[m]
    if len(matched) == 0: continue
    fc = [c for c in matched.columns if c.startswith('feat')]
    for _, row in matched.iterrows():
        idx = node_to_idx.get(row['clId'])
        if idx is not None: feat[idx] = torch.tensor(row[fc].values.astype(np.float32))

all_ids = np.array(sorted(cc_to_nodes.keys()))
all_labels = np.array([cc_label_map[c] for c in all_ids])
train_ids, temp_ids, tl, templ = train_test_split(all_ids, all_labels, test_size=0.3, stratify=all_labels, random_state=SEED)
val_ids, test_ids, vl, testl = train_test_split(temp_ids, templ, test_size=0.5, stratify=templ, random_state=SEED)

tn = []; 
for c in train_ids: tn.extend(cc_to_nodes[c])
scaler = StandardScaler()
train_feat = feat[tn].numpy()
scaler.fit(train_feat[(train_feat != 0).any(axis=1)])
feat_norm = torch.tensor(scaler.transform(feat.numpy()), dtype=torch.float)

meta = {'cc_label_map': cc_label_map, 'train_ids': train_ids.tolist(),
        'val_ids': val_ids.tolist(), 'test_ids': test_ids.tolist()}

print(f'Split: train={len(train_ids):,}, val={len(val_ids):,}, test={len(test_ids):,}')
print(f'  Train illicit: {int(tl.sum()):,}, licit: {len(train_ids)-int(tl.sum()):,}')

Features: 99it [05:33,  3.37s/it]


Split: train=85,267, val=18,271, test=18,272
  Train illicit: 1,934, licit: 83,333


## 7.5 Model + graph builder + sampler

In [91]:

class GraphSentryGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN_DIM, dropout=DROPOUT):
        super().__init__()
        self.proj = torch.nn.Linear(in_channels, hidden)
        self.conv1 = GCNConv(hidden, hidden)
        self.bn1 = torch.nn.BatchNorm1d(hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.bn2 = torch.nn.BatchNorm1d(hidden)
        self.classifier = torch.nn.Linear(hidden, 2)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.proj(x))
        h = F.relu(self.bn1(self.conv1(x, edge_index)))
        h = self.bn2(self.conv2(h, edge_index)) + x
        h = global_max_pool(h, batch)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.classifier(h)


class GraphSentryGAT(torch.nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN_DIM, dropout=DROPOUT, heads=4):
        super().__init__()
        self.proj = torch.nn.Linear(in_channels, hidden)
        self.conv1 = GATConv(hidden, hidden // heads, heads=heads)
        self.bn1 = torch.nn.BatchNorm1d(hidden)
        self.conv2 = GATConv(hidden, hidden // heads, heads=heads)
        self.bn2 = torch.nn.BatchNorm1d(hidden)
        self.classifier = torch.nn.Linear(hidden, 2)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.proj(x))
        h = F.relu(self.bn1(self.conv1(x, edge_index)))
        h = self.bn2(self.conv2(h, edge_index)) + x
        h = global_max_pool(h, batch)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.classifier(h)


MODEL_CLASSES = {'GCN': GraphSentryGCN, 'GAT': GraphSentryGAT}
INPUT_DIM = 43

model = MODEL_CLASSES[BEST_BACKBONE](INPUT_DIM).to(device)
print(f'Model: {BEST_BACKBONE}, {sum(p.numel() for p in model.parameters()):,} parameters')


def build_pyg_graph(cc_id):
    cc_nodes = sorted(cc_to_nodes[cc_id])
    local_map = {g: l for l, g in enumerate(cc_nodes)}
    x = feat_norm[cc_nodes].clone()

    node_set = set(cc_nodes)
    local_src, local_dst = [], []
    for g_src in cc_nodes:
        for g_dst in adj_list[g_src]:
            if g_dst in node_set and g_dst in local_map:
                local_src.append(local_map[g_src])
                local_dst.append(local_map[g_dst])

    if len(local_src) == 0:
        ei = torch.tensor([[0], [0]], dtype=torch.long)
    else:
        ei = torch.tensor([local_src, local_dst], dtype=torch.long)

    y = torch.tensor([cc_label_map[cc_id]], dtype=torch.long)
    return Data(x=x, edge_index=ei, y=y)


print('Building val/test graphs...')
val_data = [build_pyg_graph(c) for c in tqdm(val_ids, desc='Val')]
test_data = [build_pyg_graph(c) for c in tqdm(test_ids, desc='Test')]
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)
print(f'Val: {len(val_data)}, Test: {len(test_data)}')


class SubgraphSAINTSampler:
    def __init__(self, ids):
        self.eligible_ccs = set(ids)
        self.N = bg['num_nodes']
        self.M = bg['num_edges']
        self.eligible_illicit = [c for c in self.eligible_ccs if cc_label_map[c] == 1]
        self.eligible_licit = [c for c in self.eligible_ccs if cc_label_map[c] == 0]
        self.cc_sizes = {cc: len(ns) for cc, ns in cc_to_nodes.items()}

    def sample(self, budget, strategy):
        if strategy == 'node':
            sampled = set(random.sample(range(self.N), min(budget, self.N)))
        elif strategy == 'edge':
            ei = bg['edge_index']
            idxs = random.sample(range(self.M), min(budget, self.M))
            sampled = set()
            for i in idxs:
                sampled.add(ei[0, i].item())
                sampled.add(ei[1, i].item())
        elif strategy == 'rw':
            roots = random.sample(range(self.N), min(budget, self.N))
            sampled = set(roots)
            for r in roots:
                c = r
                for _ in range(5):
                    nb = adj_list[c]
                    if not nb: break
                    c = random.choice(nb)
                    sampled.add(c)
        else:
            raise ValueError(f'Unknown strategy: {strategy}')

        touched = {node_to_cc[n] for n in sampled if n in node_to_cc and node_to_cc[n] in self.eligible_ccs}
        if not touched:
            touched = set(random.sample(list(self.eligible_ccs), min(32, len(self.eligible_ccs))))

        ti = [c for c in touched if cc_label_map[c] == 1]
        tl = [c for c in touched if cc_label_map[c] == 0]

        if ti and tl:
            cc_ids = tl + ti * max(1, len(tl) // len(ti))
        elif not ti:
            n_inject = max(1, len(tl) // 10)
            cc_ids = tl + random.choices(self.eligible_illicit, k=min(n_inject, len(self.eligible_illicit)))
        else:
            cc_ids = list(touched)

        data_list = [build_pyg_graph(c) for c in cc_ids]
        batch = Batch.from_data_list(data_list)

        # Normalisation weights
        weights = []
        for c in cc_ids:
            sz = self.cc_sizes.get(c, 1)
            if strategy == 'node':
                p = 1 - (1 - sz / self.N) ** budget
            elif strategy == 'edge':
                d_c = sum(len(adj_list[n]) for n in cc_to_nodes[c])
                p = 1 - (1 - d_c / (2 * self.M + 1)) ** budget
            elif strategy == 'rw':
                p = 1 - (1 - sz / self.N) ** (budget * 5)
            else:
                p = 1.0
            weights.append(1.0 / max(p, 1e-6))
        w = torch.tensor(weights, dtype=torch.float)
        norm_weights = w * len(w) / w.sum()

        return batch, norm_weights

sampler = SubgraphSAINTSampler(train_ids.tolist())
print(f'Sampler ready: {len(sampler.eligible_ccs):,} train CCs')

Model: GAT, 39,938 parameters
Building val/test graphs...


Test: 100%|██████████| 18272/18272 [00:01<00:00, 10685.53it/s]


Val: 18271, Test: 18272
Sampler ready: 85,267 train CCs


## 7.6 Training

In [92]:

def weighted_focal_loss(logits, targets, norm_weights, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA):
    ce = F.cross_entropy(logits, targets, reduction='none')
    pt = torch.exp(-ce)
    fl = alpha * (1 - pt) ** gamma * ce
    return (fl * norm_weights.to(logits.device)).mean()


def evaluate(model, loader):
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            prob = F.softmax(out, dim=1)[:, 1]
            y_true.extend(batch.y.cpu().numpy())
            y_prob.extend(prob.cpu().numpy())
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    has_both = len(np.unique(y_true)) > 1
    return {
        'auroc': roc_auc_score(y_true, y_prob) if has_both else 0.0,
        'auc_pr': average_precision_score(y_true, y_prob) if has_both else 0.0,
        'f1': f1_score(y_true, (y_prob >= 0.5).astype(int), zero_division=0),
        'y_true': y_true,
        'y_prob': y_prob,
    }


def tune_threshold(y_true, y_prob):
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.1, 1.0, 0.05):
        f = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t, best_f1


optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE
)

best_val_auroc = 0
best_model_state = None
epochs_no_improve = 0
history = []

print(f'Training: {MAX_EPOCHS} max epochs, patience={PATIENCE}, '
      f'strategy={SAINT_STRATEGY}, budget={SAINT_BUDGET}')
print(f'Scale: {len(sampler.eligible_ccs):,} train CCs, {bg["num_nodes"]:,} background nodes')
print('-' * 75)

t_start = time.time()
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss = 0

    for _ in range(SAINT_SAMPLES_PER_EPOCH):
        batch, norm_weights = sampler.sample(SAINT_BUDGET, SAINT_STRATEGY)
        batch = batch.to(device)

        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = weighted_focal_loss(out, batch.y, norm_weights)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / SAINT_SAMPLES_PER_EPOCH
    val_m = evaluate(model, val_loader)
    scheduler.step(val_m['auroc'])

    if val_m['auroc'] > best_val_auroc:
        best_val_auroc = val_m['auroc']
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
        marker = ' *'
    else:
        epochs_no_improve += 1
        marker = ''

    lr_now = optimizer.param_groups[0]['lr']
    history.append({'epoch': epoch, 'loss': avg_loss,
                    'val_auroc': val_m['auroc'], 'val_f1': val_m['f1'], 'lr': lr_now})

    if epoch <= 5 or epoch % 5 == 0 or marker:
        print(f'  Epoch {epoch:3d} | Loss: {avg_loss:.4f} | '
              f"Val AUROC: {val_m['auroc']:.4f} | Val F1: {val_m['f1']:.4f} | "
              f'LR: {lr_now:.6f}{marker}')

    if epochs_no_improve >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}')
        break

t_train = time.time() - t_start
print(f'\nTraining completed in {t_train:.1f}s')
print(f'Best validation AUROC: {best_val_auroc:.4f}')

model.load_state_dict(best_model_state)

Training: 60 max epochs, patience=10, strategy=rw, budget=1000
Scale: 85,267 train CCs, 444,521 background nodes
---------------------------------------------------------------------------
  Epoch   1 | Loss: 0.2088 | Val AUROC: 0.7308 | Val F1: 0.0848 | LR: 0.005000 *
  Epoch   2 | Loss: 0.1075 | Val AUROC: 0.8694 | Val F1: 0.1246 | LR: 0.005000 *
  Epoch   3 | Loss: 0.1115 | Val AUROC: 0.8812 | Val F1: 0.1551 | LR: 0.005000 *
  Epoch   4 | Loss: 0.0911 | Val AUROC: 0.8832 | Val F1: 0.1555 | LR: 0.005000 *
  Epoch   5 | Loss: 0.0892 | Val AUROC: 0.8964 | Val F1: 0.1602 | LR: 0.005000 *
  Epoch   7 | Loss: 0.0827 | Val AUROC: 0.8975 | Val F1: 0.1618 | LR: 0.005000 *
  Epoch  10 | Loss: 0.0780 | Val AUROC: 0.9104 | Val F1: 0.1621 | LR: 0.005000 *
  Epoch  11 | Loss: 0.0767 | Val AUROC: 0.9123 | Val F1: 0.1457 | LR: 0.005000 *
  Epoch  15 | Loss: 0.0754 | Val AUROC: 0.9094 | Val F1: 0.1575 | LR: 0.005000
  Epoch  16 | Loss: 0.0763 | Val AUROC: 0.9160 | Val F1: 0.1727 | LR: 0.005000 *
  E

<All keys matched successfully>

## 7.7 Evaluation + fingerprints + save

In [93]:

val_r = evaluate(model, val_loader)
best_thresh, val_f1_tuned = tune_threshold(val_r['y_true'], val_r['y_prob'])
print(f'Optimal threshold (from val): {best_thresh:.2f} (val F1 = {val_f1_tuned:.4f})')

test_r = evaluate(model, test_loader)
y_pred = (test_r['y_prob'] >= best_thresh).astype(int)
cm = confusion_matrix(test_r['y_true'], y_pred)

print(f"\nTest results @ threshold={best_thresh:.2f}:")
print(f"  AUROC:     {test_r['auroc']:.4f}")
print(f"  AUC-PR:    {test_r['auc_pr']:.4f}")
print(f"  F1:        {f1_score(test_r['y_true'], y_pred, zero_division=0):.4f}")
print(f"  Precision: {precision_score(test_r['y_true'], y_pred, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(test_r['y_true'], y_pred, zero_division=0):.4f}")
print(f"\nConfusion matrix:")
print(f"  TN={cm[0,0]}, FP={cm[0,1]}")
print(f"  FN={cm[1,0]}, TP={cm[1,1]}")
print(f"\nScale: {len(cc_to_nodes):,} CCs, {N:,} nodes")

torch.save(best_model_state, os.path.join(PROCESSED_PATH, 'fullscale_model.pth'))

fs_results = {
    'backbone': BEST_BACKBONE,
    'strategy': SAINT_STRATEGY,
    'architecture': '2-layer with residual, max pool, focal loss',
    'input_dim': INPUT_DIM,
    'scale': {'total_ccs': len(cc_to_nodes), 'total_nodes': N,
              'total_edges': bg['num_edges'], 'train_ccs': len(train_ids),
              'val_ccs': len(val_ids), 'test_ccs': len(test_ids)},
    'threshold': best_thresh,
    'test_auroc': round(test_r['auroc'], 4),
    'test_auc_pr': round(test_r['auc_pr'], 4),
    'test_f1': round(f1_score(test_r['y_true'], y_pred, zero_division=0), 4),
    'test_precision': round(precision_score(test_r['y_true'], y_pred, zero_division=0), 4),
    'test_recall': round(recall_score(test_r['y_true'], y_pred, zero_division=0), 4),
    'training_time_s': round(t_train, 1),
    'epochs_run': len(history),
}

with open(os.path.join(PROCESSED_PATH, 'fullscale_results.json'), 'w') as f:
    json.dump(fs_results, f, indent=2)

print('\nComputing fingerprints for test set...')
fp_list = []
for i, data in enumerate(tqdm(test_data, desc='Fingerprints')):
    G = nx.Graph()
    ei = data.edge_index.numpy()
    for s, d in zip(ei[0], ei[1]):
        G.add_edge(int(s), int(d))
    n_nodes = data.num_nodes
    n_edges = data.num_edges
    degs = [d for _, d in G.degree()] if G.number_of_nodes() > 0 else [0]
    fp_list.append({
        'nodes': n_nodes,
        'edges': n_edges,
        'edge_node_ratio': n_edges / max(n_nodes, 1),
        'deg_mean': np.mean(degs),
        'deg_std': np.std(degs),
        'deg_max': max(degs),
        'density': nx.density(G) if G.number_of_nodes() > 1 else 0,
        'clustering': nx.average_clustering(G) if G.number_of_nodes() > 1 else 0,
        'label': data.y.item(),
        'risk_score': float(test_r['y_prob'][i]),
    })

fp_arr = np.array([[fp['nodes'], fp['edges'], fp['edge_node_ratio'], fp['deg_mean'],
                     fp['deg_std'], fp['deg_max'], fp['density'], fp['clustering']]
                    for fp in fp_list])
labels_arr = np.array([fp['label'] for fp in fp_list])
risks_arr = np.array([fp['risk_score'] for fp in fp_list])
nodes_arr = np.array([fp['nodes'] for fp in fp_list])
edges_arr = np.array([fp['edges'] for fp in fp_list])

np.savez(os.path.join(PROCESSED_PATH, 'fullscale_fingerprints.npz'),
         fingerprints=fp_arr, labels=labels_arr, risk_scores=risks_arr,
         nodes=nodes_arr, edges=edges_arr)

print(f"Saved {len(fp_list)} fingerprints")
print(f"\nSaved fullscale_model.pth")
print(f"Saved fullscale_results.json")
print(f"\nFull-scale training complete.")
print(f"Test AUROC: {test_r['auroc']:.4f}")

Optimal threshold (from val): 0.75 (val F1 = 0.4876)

Test results @ threshold=0.75:
  AUROC:     0.9202
  AUC-PR:    0.4610
  F1:        0.4746
  Precision: 0.6235
  Recall:    0.3831

Confusion matrix:
  TN=17761, FP=96
  FN=256, TP=159

Scale: 121,810 CCs, 444,521 nodes

Computing fingerprints for test set...


Fingerprints: 100%|██████████| 18272/18272 [00:04<00:00, 4175.15it/s]

Saved 18272 fingerprints

Saved fullscale_model.pth
Saved fullscale_results.json

Full-scale training complete.
Test AUROC: 0.9202
